# 01. Actualización monetaria mediante INPC

Se actualizan importes anuales a diciembre de 2025.

**Pasos:**

- Obtener los índices del INPC (se usa el API del banco de mexico mediante la libreria banxicolab)

- Actualizar primas y comisiones a diciembre del 2025

- Actualizar siniestros según el año de pago

- Guardar snapshots de las bases ajustadas 

- Documentar el proceso de actualización

In [21]:
# ------------------------------------------
# 1. Importación de librerías
# ------------------------------------------

from pathlib import Path
import sys
import pandas as pd

# ------------------------------------------
# 2. Configuración de rutas
# ------------------------------------------

# Localizar la carpeta principal del proyecto
ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "funciones_reservas.py").exists()
)

# Permitir importar las funciones del proyecto
sys.path.insert(0, str(ROOT))

from funciones_reservas import (
    ajustar_inpc,
    guardar_snapshot,
    normalizar_categorias
)

# Definir las carpetas de datos
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"

# ------------------------------------------
# 3. Importación de bases de datos
# ------------------------------------------

# Leer las hojas de emisión y siniestralidad
archivo = RAW / "2026 BASE RESERVAS.xlsx"

emision = pd.read_excel(archivo, sheet_name="EMISION")
siniestros = pd.read_excel(archivo, sheet_name="SINIESTRALIDAD")

# ------------------------------------------
# 4. Normalización de categorías
# ------------------------------------------

# Unificar la escritura de las variables categóricas
columnas = ["tipo_embarcacion", "uso"]

for df in [emision, siniestros]:
    df[columnas] = normalizar_categorias(df, columnas)[columnas]


## Serie INPC y criterio temporal

In [22]:
# ------------------------------------------
#  Descargar serie INPC mensual
# ------------------------------------------

import banxicolab as sie

# SE REQUIERE CONFIGURAR EL TOKEN DE BANXICO 
# sie.configurar(token="TU_TOKEN_AQUI")

serie_inpc = pd.Series(sie.descargar("inpc", inicio="2015-01-01", fin="2025-12-31")["inpc"])


### 1.1 Ajustar los valores Mediante el INPC

Pasos:
1. Configurar el token de Banxico 
2. Descargar la serie mensual del INPC
3. Ajustar los valores mediante el INPC


La fecha objetivo es diciembre de 2025; para registros anuales se usa diciembre del a?o de origen.

In [23]:
# --------------------------------------------------
# 1.2.3 Actualización de cifras mediante el INPC
# --------------------------------------------------

# Actualizar primas y comisiones según su año de emisión
emision_ajustada = ajustar_inpc(
    emision,
    ["prima_emitida", "prima_retenida", "comision_directa"],
    "año", 2025, serie_inpc
)


# Actualizar los montos de siniestros según su año de pago
siniestros_ajustados = siniestros.copy()

for desarrollo in range(4):

    columna = f"{desarrollo}_monto pagado"

    # Calcular el año correspondiente a cada desarrollo
    siniestros_ajustados["año_pago"] = siniestros["año"] + desarrollo

    # Considerar únicamente los pagos hasta 2025
    mascara = siniestros_ajustados["año_pago"] <= 2025

    # Ajustar los montos del desarrollo correspondiente
    datos = ajustar_inpc(
        siniestros_ajustados.loc[mascara, ["año_pago", columna]],
        columna, "año_pago", 2025, serie_inpc
    )

    siniestros_ajustados[f"{columna}_ajustada"] = datos[f"{columna}_ajustada"]

# Eliminar la columna temporal
siniestros_ajustados.drop(columns="año_pago", inplace=True)


# --------------------------------------------------
# 1.2.4 Guardar snapshots de las bases ajustadas
# --------------------------------------------------

# Guardar la base de emisión
guardar_snapshot(
    emision_ajustada,
    PROCESSED / "emision_ajustada_2025.parquet",
    periodo_valuacion="2025-12",
    fuente_inpc="Banxico",
    criterio_actualizacion="INPC de diciembre de cada año",
    archivos_origen=[archivo]
)

# Guardar la base de siniestros
guardar_snapshot(
    siniestros_ajustados,
    PROCESSED / "siniestros_ajustados_2025.parquet",
    periodo_valuacion="2025-12",
    fuente_inpc="Banxico",
    criterio_actualizacion="INPC de diciembre del año de pago",
    archivos_origen=[archivo]
)

# Mostrar una vista previa de los resultados
display(emision_ajustada.head(), siniestros_ajustados.head())

,año,tipo_embarcacion,uso,anio_construccion,prima_emitida,prima_retenida,comision_directa,factor_inpc,prima_emitida_ajustada,prima_retenida_ajustada,comision_directa_ajustada
0,2015,lancha,comercial,2007,3219,3219,644,1.606368,5170.900076,5170.900076,1034.501289
1,2015,buque-tanque,comercial,1992,332844,332844,66569,1.606368,534670.104079,534670.104079,106934.342090
2,2015,velero,privado y/o placer,2012,25759,25759,5151,1.606368,41378.445190,41378.445190,8274.403943
3,2015,abastecedor,privado y/o placer,2001,-37,-37,-7,1.606368,-59.435633,-59.435633,-11.244579
4,2015,lancha,pesca,2013,-194,-194,-31,1.606368,-311.635481,-311.635481,-49.797422


,año,tipo_embarcacion,uso,anio_construccion,cobertura,causa_siniestro,0_monto pagado,1_monto pagado,2_monto pagado,3_monto pagado,0_monto pagado_ajustada,1_monto pagado_ajustada,2_monto pagado_ajustada,3_monto pagado_ajustada
0,2015,abastecedor,comercial,1979.0,Todo Riesgo (Casco),Incendio,29665062.0,0.0,0.0,0.0,4.765302e+07,0.0,0.0,0.0
1,2015,abastecedor,comercial,2011.0,Todo Riesgo (Casco),Averia particular,15502388.0,0.0,0.0,0.0,2.490255e+07,0.0,0.0,0.0
2,2015,buque-tanque,comercial,2013.0,Todo Riesgo (Casco),Varadura,10377414.0,0.0,0.0,0.0,1.666995e+07,0.0,0.0,0.0
3,2015,buque-tanque,comercial,2014.0,Todo Riesgo (Casco),Negligencia de la tripulacion,9721320.0,0.0,0.0,0.0,1.561602e+07,0.0,0.0,0.0
4,2015,remolcador,remolcador,2013.0,Todo Riesgo (Casco),Averia particular,9668053.0,0.0,0.0,0.0,1.553046e+07,0.0,0.0,0.0


Para actualizar las cifras monetarias, se utilizó la API del Banco de México mediante la librería `banxicolab`, la cual requiere configurar un token personal para consultar la serie histórica del Índice Nacional de Precios al Consumidor (INPC). Una vez obtenidos los índices, se calcularon los factores de actualización para expresar los montos de cada año en pesos de diciembre de 2025.